# Global Solution 1ºSemestre 2026
## Indústria Espacial: O Código que Move o Universo

O monitoramento contínuo da cobertura terrestre é crucial para mitigar os impactos das mudanças climáticas e guiar o desenvolvimento urbano sustentável. No entanto, a análise manual de imagens de satélite é impraticável devido ao volume massivo de dados gerados diariamente. Este trabalho utiliza o dataset EuroSAT para treinar uma Rede Neural Convolucional (CNN) capaz de classificar automaticamente o uso do solo em 10 categorias distintas. A automação desse processo permite que órgãos governamentais e ambientais tomem decisões rápidas baseadas em dados geoespaciais atualizados, transformando imagens brutas de satélite em inteligência ambiental acionável.

## Importações necessárias

In [1]:
import torch
from torch import nn
import torchvision
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, random_split

In [2]:
device = torch.device("cuda"if torch.cuda.is_available else "cpu")
print(f"Device = {device}")

Device = cuda


## Preparando dataset de imagens EuroSAT

=> Separação dos dados de treino, teste e validação
- treino = 80%
- teste = 10%
- validação = 10%

In [3]:
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

path_dados = "./dataset/EuroSAT_RGB"

dataset_completo = datasets.ImageFolder(
    root=path_dados,
    transform=transform
)

print(f"Total de imagens carregadas: {len(dataset_completo)}")
print(f"Classes disponíveis: {dataset_completo.classes}")

tamanho_total = len(dataset_completo)
tam_treino = int(0.8 * tamanho_total)
tam_val = int(0.1 * tamanho_total)
tam_teste = tamanho_total - tam_treino - tam_val

dataset_treino, dataset_val, dataset_teste = random_split(
    dataset_completo, [tam_treino, tam_val, tam_teste], 
    generator = torch.Generator().manual_seed(42)
)

batch_size = 32 
loader_treino = DataLoader(dataset_treino, batch_size=batch_size, shuffle=True)
loader_val = DataLoader(dataset_val, batch_size=batch_size, shuffle=False)
loader_teste = DataLoader(dataset_teste, batch_size=batch_size, shuffle=False)

print("Dataset pronto para o treinamento!")




Total de imagens carregadas: 27000
Classes disponíveis: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Dataset pronto para o treinamento!


## Primeira Arquitetura CNN

In [4]:
class CNNBaseline(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model1 = CNNBaseline(num_classes=10).to(device)
print(model1)


CNNBaseline(
  (features): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=4096, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=10, bias=True)
  )
)


# Segunda Arquitetura CNN

In [5]:
class CNNAvancada(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.4)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x
    
model2 = CNNAvancada(num_classes=10).to(device)
print(model2)

CNNAvancada(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (5): ReLU()
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): Dropout(p=0.25, inplace=False)
    (8): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (10): ReLU()
    (11): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (12): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (13): ReLU()
    (14): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ce

## Criando função de treino para uma época

In [6]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        # 1. Zera os gradientes acumulados
        optimizer.zero_grad()

        # 2. Forward: calcula a saída do modelo
        logits = model(images)

        # 3. Calcula o erro
        loss = criterion(logits, labels)

        # 4. Backward: calcula os gradientes
        loss.backward()

        # 5. Atualiza os pesos
        optimizer.step()

        # Métricas simples
        total_loss += loss.item() # Acumula a perda do batch
        preds = torch.argmax(logits, dim=1) # Obtém as classes previstas
        correct += (preds == labels).sum().item() # Conta acertos
        total += labels.size(0) # Conta o total de amostras processadas

    avg_loss = total_loss / len(dataloader) # Calcula a perda média do epoch
    accuracy = (correct / total) * 100 # Calcula a acurácia do epoch

    return avg_loss, accuracy

## Criando função de avaliação

In [7]:
def evaluate(model, dataloader, criterion, device):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)
            loss = criterion(logits, labels)

            total_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = (correct / total) * 100

    return avg_loss, accuracy

## Criando função genérica de treinamento

In [ ]:
def treinar_modelo(modelo, train_loader, test_loader, val_loader, criterion, optimizer, epochs=15, model_num=1):
    
    best_val_loss = float('inf')
    wait_count = 0
    patience = 5   # Early stop acionado depois de 5 epochs sem melhoras a perda da validacao
    
    historico = {
        'train_loss': [], 'train_acc': [],
        'test_loss': [], 'test_acc': [],
        'val_loss': [], 'val_acc': []
    }

    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(modelo, train_loader, criterion, optimizer, device)
        test_loss, test_acc = evaluate(modelo, test_loader, criterion, device)
        val_loss, val_acc = evaluate(modelo, val_loader, criterion, device)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            wait_count = 0
            torch.save(modelo.state_dict(), f'best_model{model_num}_weights.pth')
        else:
            wait_count += 1
            print(f"⚠️ Atenção: Validation Loss não melhorou por {wait_count} épocas.")
        
        if wait_count >= patience:
            print("\n🛑 Early Stopping Acionado!") # Early stop aplicado para evitar overfitting, salvando os pesos na melhor epoch
            break


        historico['train_loss'].append(train_loss)
        historico['train_acc'].append(train_acc)
        historico['test_loss'].append(test_loss)
        historico['test_acc'].append(test_acc)
        historico['val_loss'].append(val_loss)
        historico['val_acc'].append(val_acc)

        print(f"Época [{epoch+1}/{epochs}] | "
              f"Treino Loss: {train_loss:.4f} Treino Acc: {train_acc:.2f}% | "
              f"Teste Loss: {test_loss:.4f} Teste Acc: {test_acc:.2f}% | "
              f"Valid. Loss: {val_loss:.4f} Valid. Acc: {val_acc:.2f}%"
              )
    
    return historico

## Treinamento das 2 arquiteturas criadas

In [9]:
criterion = nn.CrossEntropyLoss()

print("Treinando modelo 1")
otimizador1 = torch.optim.Adam(model1.parameters(), lr=0.001)
historico1 = treinar_modelo(model1, loader_treino, loader_teste, loader_val, criterion, otimizador1)

print("Treinando modelo 2")
otimizador2 = torch.optim.Adam(model2.parameters(), lr=0.001, weight_decay=1e-4)
historico2 = treinar_modelo(model2, loader_treino, loader_teste, loader_val, criterion, otimizador2, epochs=30, model_num=2)

Treinando modelo 1
Época [1/15] | Treino Loss: 0.9935 Treino Acc: 62.59% | Teste Loss: 0.6831 Teste Acc: 75.15% | Valid. Loss: 0.6814 Valid. Acc: 75.19%
Época [2/15] | Treino Loss: 0.6056 Treino Acc: 78.10% | Teste Loss: 0.5232 Teste Acc: 80.89% | Valid. Loss: 0.5221 Valid. Acc: 82.15%
Época [3/15] | Treino Loss: 0.4852 Treino Acc: 82.90% | Teste Loss: 0.5105 Teste Acc: 81.70% | Valid. Loss: 0.4864 Valid. Acc: 82.70%
Época [4/15] | Treino Loss: 0.4108 Treino Acc: 85.53% | Teste Loss: 0.4112 Teste Acc: 84.89% | Valid. Loss: 0.4120 Valid. Acc: 85.63%
Época [5/15] | Treino Loss: 0.3464 Treino Acc: 87.71% | Teste Loss: 0.3452 Teste Acc: 87.56% | Valid. Loss: 0.3588 Valid. Acc: 87.41%
⚠️ Atenção: Validation Loss não melhorou por 1 épocas.
Época [6/15] | Treino Loss: 0.2858 Treino Acc: 90.00% | Teste Loss: 0.3490 Teste Acc: 87.52% | Valid. Loss: 0.3736 Valid. Acc: 87.30%
⚠️ Atenção: Validation Loss não melhorou por 2 épocas.
Época [7/15] | Treino Loss: 0.2340 Treino Acc: 91.81% | Teste Loss:

## Conclusão dos treinamentos

No treinamento do modelo 1, podemos observar o seguinte comportamento em sua melhor epoch:

- Pontuação da acurácia de treino == validação (87,71%), parado com o early stop para evitar overfitting;
- Perda de validação ligeiramente alta (0.35).

Ao observar o treinamento do modelo 2, conseguimos notar que foi encontrado um equilíbrio saudável com uma melhor performance comparado ao modelo 1, ao analisar os seguintes pontos:

- Pontuação da acurácia de treino (95.83%) muito próxima a acurácia de validação (95.59%), com early stop evitando overfitting;
- Perda de validação baixa (0.12).

Portanto, concluímos que o modelo 2 é o ideal para ser colocado em produção! Utilizaremos no deploy o arquivos de pesos "best_model2_weights.pth"